# Hybrid IDS: Isolation Forest + XGBoost + Graph-Based Heuristic Search

Code accompanying the paper *"A Hybrid Network Intrusion Detection and Post-Breach Network Analysis Framework Using Isolation Forest, XGBoost, and Graph-Based Heuristic Search on CICIDS-2017."*

## Data setup

This notebook expects the five CICIDS-2017 daily CSV files in a `data/` folder next to the notebook:

```
data/monday.csv
data/tuesday.csv
data/wednesday.csv
data/thursday.csv
data/friday.csv
```

The dataset is available from the Canadian Institute for Cybersecurity:
https://www.unb.ca/cic/datasets/ids-2017.html

## Notebook structure

**Cells below, up to the "Loading all CSV files" cell, are initial exploratory work on a single day's capture.** They are kept for transparency but are *not* the pipeline reported in the paper.

**The pipeline reported in the paper begins at the "Loading all CSV files" cell**, which combines all five days into the full 2,099,971-sample dataset. Everything from that point onward — preprocessing, the two-stage model, cross-validation, both ablation studies, and the graph analysis — corresponds to the results in the paper.

Outputs have been cleared to keep the file readable. Reported figures are listed in the repository README.


In [ ]:
!pip install pandas numpy matplotlib scikit-learn xgboost

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import networkx as nx
from collections import deque

with open('model_weighted.pkl', 'rb') as f:
    model_weighted = pickle.load(f)
with open('le_combined.pkl', 'rb') as f:
    le_combined = pickle.load(f)
with open('scaler_combined.pkl', 'rb') as f:
    scaler_combined = pickle.load(f)

print("All libraries and models loaded!")

In [ ]:
import xgboost as xgb
print("XGBoost version:", xgb.__version__)

try:
    test_model = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=5)
    import numpy as np
    test_model.fit(np.random.rand(100, 5), np.random.randint(0, 2, 100))
    print("GPU acceleration is working -- cells below will use your RTX 4050.")
except Exception as e:
    print("GPU acceleration failed, will fall back to CPU. Error:")
    print(e)
    print("\nIf this fails: check 'nvidia-smi' works in a terminal (driver installed),")
    print("and that you installed the GPU-enabled xgboost wheel (the default pip")
    print("install usually includes CUDA support on Windows, but worth confirming).")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/friday.csv")

print("Shape:", df.shape)
print("\nLabel distribution:")
print(df['Label'].value_counts())

In [ ]:
print("Missing values per column:")
print(df.isnull().sum().sum(), "total missing values")

print("\nFirst 5 rows:")
df.head()

In [ ]:
df.columns = df.columns.str.strip()

print("Cleaned column names:")
print(df.columns.tolist())

In [ ]:
df = df.drop(columns=['Src IP dec', 'Dst IP dec', 'Timestamp'])

print("Shape after dropping irrelevant columns:", df.shape)

print("\nInfinite values:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())

In [ ]:
plt.figure(figsize=(10, 6))
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6', '#3498db']

df['Label'].value_counts().plot(
    kind='bar',
    color=colors,
    edgecolor='black'
)

plt.title('Class Distribution in CICIDS-2017 Friday Dataset', fontsize=14)
plt.xlabel('Attack Type', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=300)
plt.show()

print("Plot saved!")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

X = df.drop(columns=['Label'])
y = df['Label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Label encoding:")
for i, label in enumerate(le.classes_):
    print(f"  {label} → {i}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nScaling complete!")
print("X shape:", X_scaled.shape)

In [ ]:
from sklearn.ensemble import IsolationForest

print("Training Isolation Forest...")
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.47,  
    random_state=42,
    n_jobs=-1  
)

iso_forest.fit(X_scaled)

iso_predictions = iso_forest.predict(X_scaled)

iso_binary = (iso_predictions == -1).astype(int)

print("Done!")
print("Anomalies detected:", iso_binary.sum())
print("Normal traffic:", (iso_binary == 0).sum())

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

iso_scores = iso_forest.score_samples(X_scaled)
X_final = np.column_stack([X_scaled, iso_scores])

print("Final feature matrix shape:", X_final.shape)
print("\nTraining XGBoost with Stratified K-Fold...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_final, y_encoded)):
    X_train, X_test = X_final[train_idx], X_final[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
    
    model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        reg_alpha=0.1,   # L1
        reg_lambda=1.0,  # L2
        random_state=42,
        n_jobs=-1,
        eval_metric='mlogloss'
    )
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    report = classification_report(y_test, y_pred, 
                                  target_names=le.classes_, 
                                  output_dict=True)
    fold_results.append(report)
    print(f"Fold {fold+1} complete ✓")

print("\nAll folds complete!")

In [ ]:
metrics = ['precision', 'recall', 'f1-score']
classes = le.classes_

print("=" * 60)
print("AVERAGE RESULTS ACROSS 5 FOLDS")
print("=" * 60)

for cls in classes:
    avg_f1 = np.mean([f[cls]['f1-score'] for f in fold_results])
    avg_prec = np.mean([f[cls]['precision'] for f in fold_results])
    avg_rec = np.mean([f[cls]['recall'] for f in fold_results])
    print(f"\n{cls}:")
    print(f"  Precision: {avg_prec:.4f}")
    print(f"  Recall:    {avg_rec:.4f}")
    print(f"  F1-Score:  {avg_f1:.4f}")

avg_accuracy = np.mean([f['accuracy'] for f in fold_results])
print(f"\n{'='*60}")
print(f"Overall Accuracy: {avg_accuracy:.4f}")

In [ ]:
!pip install seaborn

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred_final = model.predict(X_test)

plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)

plt.title('Confusion Matrix - XGBoost (Final Fold)', fontsize=14)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

print("Confusion matrix saved!")

In [ ]:
feature_names = list(df.drop(columns=['Label']).columns) + ['Anomaly_Score']

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(importance_df['Feature'][::-1], 
         importance_df['Importance'][::-1],
         color='steelblue', edgecolor='black')
plt.title('Top 20 Most Important Features - XGBoost', fontsize=14)
plt.xlabel('Feature Importance Score', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.show()

print("Feature importance saved!")

---

# Paper pipeline starts here

Everything below corresponds to the methodology and results reported in the paper: all five days combined (2,099,971 samples, 27 traffic classes), leakage-free Stratified 5-Fold cross-validation, both ablation studies, and the graph-based post-detection analysis.


In [ ]:
import pandas as pd
import os

print("Loading all CSV files...")

df_friday = pd.read_csv("data/friday.csv")
df_wednesday = pd.read_csv("data/wednesday.csv")
df_tuesday = pd.read_csv("data/tuesday.csv")
df_thursday = pd.read_csv("data/thursday.csv")
df_monday = pd.read_csv("data/monday.csv")

df_combined = pd.concat([df_friday, df_wednesday, df_tuesday, df_thursday, df_monday], ignore_index=True)

print("Individual shapes:")
print(f"  Friday:    {df_friday.shape}")
print(f"  Wednesday: {df_wednesday.shape}")
print(f"  Tuesday:   {df_tuesday.shape}")
print(f"  Thursday:  {df_thursday.shape}")
print(f"  Monday:    {df_monday.shape}")
print(f"\nCombined shape: {df_combined.shape}")
print("\nCombined label distribution:")
print(df_combined['Label'].value_counts())

In [ ]:
df_combined.columns = df_combined.columns.str.strip()
df_combined = df_combined.drop(columns=['Src IP dec', 'Dst IP dec', 'Timestamp'])
df_combined = df_combined.replace([np.inf, -np.inf], np.nan).dropna()

print("Shape after cleaning:", df_combined.shape)
print("Total labels:", df_combined['Label'].nunique(), "unique classes")

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

print("Attempted Category value counts:")
print(df_combined['Attempted Category'].value_counts())
print("\nCrosstab against Label (attack classes only):")
print(pd.crosstab(df_combined['Attempted Category'], df_combined['Label']))

DROP_ATTEMPTED_CATEGORY = True

X_combined = df_combined.drop(columns=['Label'])
if DROP_ATTEMPTED_CATEGORY and 'Attempted Category' in X_combined.columns:
    X_combined = X_combined.drop(columns=['Attempted Category'])
    print("\n'Attempted Category' dropped from feature matrix.")

y_combined = df_combined['Label']

le_combined = LabelEncoder()
y_combined_encoded = le_combined.fit_transform(y_combined)

print("\nLabel encoding (27 classes):")
for i, label in enumerate(le_combined.classes_):
    print(f"  {label} -> {i}")

X_combined_raw = X_combined.values  

print("\nRaw (unscaled) feature matrix shape:", X_combined_raw.shape)
print("Scaling will be performed per-fold, not globally.")


In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

print("[Reference only -- not used in final pipeline] Training a global Isolation Forest for a quick look...")
_iso_preview = IsolationForest(
    n_estimators=100,
    contamination=0.25,
    random_state=42,
    n_jobs=-1
)
_iso_preview.fit(X_combined_raw)
_iso_scores_preview = _iso_preview.score_samples(X_combined_raw)
print("Preview anomaly score range:", _iso_scores_preview.min(), "to", _iso_scores_preview.max())
print("(Real Isolation Forest fitting for results happens per-fold in the next cell.)")


In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import numpy as np

print("Retraining XGBoost with class weights + Stratified K-Fold...")
print("Using XGBoost's native Booster API (not XGBClassifier) -- this avoids a")
print("known XGBoost limitation where a fold missing a rare class entirely from")
print("its training split raises 'Invalid classes inferred from unique values of y'.")

NUM_CLASSES = len(le_combined.classes_)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results_weighted = []
fold_feature_importances = []
last_booster = None
last_X_test_final = None
last_y_test = None

xgb_params = {
    'objective': 'multi:softprob',
    'num_class': NUM_CLASSES,
    'max_depth': 6,
    'eta': 0.1,          
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'seed': 42,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda',    
}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_combined_raw, y_combined_encoded)):
    X_train_raw, X_test_raw = X_combined_raw[train_idx], X_combined_raw[test_idx]
    y_train, y_test = y_combined_encoded[train_idx], y_combined_encoded[test_idx]

    scaler_fold = StandardScaler()
    X_train_scaled = scaler_fold.fit_transform(X_train_raw)
    X_test_scaled = scaler_fold.transform(X_test_raw)

    iso_fold = IsolationForest(n_estimators=100, contamination=0.25, random_state=42, n_jobs=-1)
    iso_fold.fit(X_train_scaled)
    train_iso_scores = iso_fold.score_samples(X_train_scaled).reshape(-1, 1)
    test_iso_scores = iso_fold.score_samples(X_test_scaled).reshape(-1, 1)

    X_train_final = np.hstack([X_train_scaled, train_iso_scores])
    X_test_final = np.hstack([X_test_scaled, test_iso_scores])

    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    dtrain = xgb.DMatrix(X_train_final, label=y_train, weight=sample_weights)
    dtest = xgb.DMatrix(X_test_final, label=y_test)

    booster = xgb.train(xgb_params, dtrain, num_boost_round=100)

    y_pred_proba = booster.predict(dtest)
    y_pred = y_pred_proba.argmax(axis=1)

    report = classification_report(y_test, y_pred,
                                    labels=list(range(NUM_CLASSES)),
                                    target_names=le_combined.classes_,
                                    output_dict=True, zero_division=0)
    fold_results_weighted.append(report)

    importance_dict = booster.get_score(importance_type='gain')
    n_features = X_train_final.shape[1]
    importance_vec = np.zeros(n_features)
    for k, v in importance_dict.items():
        idx = int(k[1:])  # keys look like 'f0', 'f1', ...
        importance_vec[idx] = v
    fold_feature_importances.append(importance_vec)

    last_booster = booster
    last_X_test_final = X_test_final
    last_y_test = y_test

    print(f"Fold {fold+1} complete (leakage-free: scaler + IF both refit on train only)")

print("\nAll folds complete!")

avg_feature_importance = np.mean(fold_feature_importances, axis=0)
print("\nFeature importance is now averaged across all 5 folds (see next cells).")


In [ ]:
print("=" * 60)
print("AVERAGE RESULTS - WEIGHTED MODEL")
print("=" * 60)

for cls in le_combined.classes_:
    try:
        avg_f1 = np.mean([f[cls]['f1-score'] for f in fold_results_weighted])
        avg_prec = np.mean([f[cls]['precision'] for f in fold_results_weighted])
        avg_rec = np.mean([f[cls]['recall'] for f in fold_results_weighted])
        print(f"\n{cls}:")
        print(f"  Precision: {avg_prec:.4f}")
        print(f"  Recall:    {avg_rec:.4f}")
        print(f"  F1-Score:  {avg_f1:.4f}")
    except:
        print(f"\n{cls}: insufficient samples")

avg_accuracy = np.mean([f['accuracy'] for f in fold_results_weighted])
print(f"\n{'='*60}")
print(f"Overall Accuracy: {avg_accuracy:.4f}")

In [ ]:
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

print("Fitting final deployment model on the FULL dataset...")

final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X_combined_raw)

final_iso = IsolationForest(n_estimators=100, contamination=0.25, random_state=42, n_jobs=-1)
final_iso.fit(X_all_scaled)
final_iso_scores = final_iso.score_samples(X_all_scaled).reshape(-1, 1)

X_final_combined = np.hstack([X_all_scaled, final_iso_scores])

final_sample_weights = compute_sample_weight(class_weight='balanced', y=y_combined_encoded)

final_model_weighted = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1,
    eval_metric='mlogloss',
    tree_method='hist',
    device='cuda'  
)
final_model_weighted.fit(X_final_combined, y_combined_encoded, sample_weight=final_sample_weights)

with open('model_weighted.pkl', 'wb') as f:
    pickle.dump(final_model_weighted, f)
with open('le_combined.pkl', 'wb') as f:
    pickle.dump(le_combined, f)
with open('scaler_combined.pkl', 'wb') as f:
    pickle.dump(final_scaler, f)
with open('iso_combined.pkl', 'wb') as f:
    pickle.dump(final_iso, f)

print("All models saved! (final_model_weighted trained on full data, separate from CV evaluation)")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

dtest_final = xgb.DMatrix(last_X_test_final)
y_pred_proba_final = last_booster.predict(dtest_final)
y_pred_final = y_pred_proba_final.argmax(axis=1)

plt.figure(figsize=(18, 14))
cm = confusion_matrix(last_y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_combined.classes_,
            yticklabels=le_combined.classes_)
plt.title('Confusion Matrix -- Final Fold (Leakage-Free)', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix_weighted.png', dpi=300)
plt.show()
print("Saved!")


In [ ]:
feature_names = list(X_combined.columns) + ['Anomaly_Score']

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': avg_feature_importance
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(importance_df['Feature'][::-1],
         importance_df['Importance'][::-1],
         color='steelblue', edgecolor='black')
plt.title('Top 20 Most Important Features - Weighted XGBoost (Averaged Across 5 Folds)', fontsize=14)
plt.xlabel('Feature Importance Score (mean across folds)', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance_weighted.png', dpi=300)
plt.show()

print("Saved! (Note: rank/values will differ slightly from the single-fold version --")
print(" re-check which features land in the top 5 before restating the Isolation")
print(" Forest 'ranked 5th' claim in the paper.)")


In [ ]:
!pip install networkx

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque

G = nx.DiGraph()

nodes = {
    'Internet': {'type': 'external', 'risk': 0.0},
    'Firewall': {'type': 'security', 'risk': 0.0},
    'Router': {'type': 'infrastructure', 'risk': 0.0},
    'WebServer': {'type': 'server', 'risk': 0.0},
    'DBServer': {'type': 'server', 'risk': 0.0},
    'FileServer': {'type': 'server', 'risk': 0.0},
    'Workstation1': {'type': 'endpoint', 'risk': 0.0},
    'Workstation2': {'type': 'endpoint', 'risk': 0.0},
    'Workstation3': {'type': 'endpoint', 'risk': 0.0},
}

for node, attrs in nodes.items():
    G.add_node(node, **attrs)

edges = [
    ('Internet', 'Firewall'),
    ('Firewall', 'Router'),
    ('Router', 'WebServer'),
    ('Router', 'FileServer'),
    ('WebServer', 'DBServer'),
    ('Router', 'Workstation1'),
    ('Router', 'Workstation2'),
    ('Router', 'Workstation3'),
    ('Workstation1', 'FileServer'),
    ('Workstation2', 'FileServer'),
    ('Workstation3', 'DBServer'),
]

G.add_edges_from(edges)

print("Network graph built!")
print(f"Nodes: {list(G.nodes())}")
print(f"Edges: {G.number_of_edges()} connections")

In [ ]:
plt.figure(figsize=(14, 10))

pos = {
    'Internet': (0, 2),
    'Firewall': (2, 2),
    'Router': (4, 2),
    'WebServer': (6, 3),
    'FileServer': (6, 2),
    'DBServer': (8, 3),
    'Workstation1': (6, 1),
    'Workstation2': (6, 0),
    'Workstation3': (6, -1),
}

color_map = {
    'external': '#e74c3c',      
    'security': '#e67e22',     
    'infrastructure': '#f1c40f', 
    'server': '#3498db',        
    'endpoint': '#2ecc71',      
}

node_colors = [color_map[G.nodes[n]['type']] for n in G.nodes()]

nx.draw_networkx(G, pos,
                node_color=node_colors,
                node_size=2000,
                font_size=8,
                font_weight='bold',
                arrows=True,
                edge_color='gray',
                arrowsize=20)

plt.title('Network Topology Graph', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('network_graph.png', dpi=300)
plt.show()

print("Network graph saved!")

In [ ]:
def bfs_blast_radius(graph, start_node, infected_node, attack_type):
    
    visited = []
    queue = deque([start_node])
    level = {start_node: 0}
    
    print(f"{'='*55}")
    print(f"BFS BLAST RADIUS ANALYSIS")
    print(f"Attack Type Detected: {attack_type}")
    print(f"Entry Point: {start_node}")
    print(f"Infected Node: {infected_node}")
    print(f"{'='*55}")
    
    while queue:
        node = queue.popleft()
        if node not in visited:
            visited.append(node)
            risk = round(1.0 - (level[node] * 0.2), 2)
            risk = max(0.1, risk)
            G.nodes[node]['risk'] = risk
            print(f"Level {level[node]} | Node: {node:<15} | Risk Score: {risk}")
            
            for neighbor in graph.neighbors(node):
                if neighbor not in visited:
                    queue.append(neighbor)
                    level[neighbor] = level[node] + 1
    
    print(f"\nTotal nodes at risk: {len(visited)}")
    print(f"Blast radius: {max(level.values())} hops from entry point")
    return visited

bfs_result = bfs_blast_radius(G, 'Internet', 'Router', 'DDoS')

In [ ]:
def dfs_attack_path(graph, start_node, target_node, attack_type):
    
    visited = []
    path = []
    
    def dfs_recursive(node):
        visited.append(node)
        path.append(node)
        
        print(f"Visiting: {node:<15} | Depth: {len(path)-1} | Path: {' → '.join(path)}")
        
        if node == target_node:
            print(f"\n*** TARGET REACHED: {target_node} ***")
            return True
            
        for neighbor in graph.neighbors(node):
            if neighbor not in visited:
                if dfs_recursive(neighbor):
                    return True
                    
        path.pop()
        return False
    
    print(f"{'='*55}")
    print(f"DFS ATTACK PATH ANALYSIS")
    print(f"Attack Type Detected: {attack_type}")
    print(f"Entry Point: {start_node}")
    print(f"Target Node: {target_node}")
    print(f"{'='*55}")
    
    dfs_recursive(start_node)
    
    print(f"\nAttack chain depth: {len(path)-1} hops")
    print(f"Full attack path: {' → '.join(path)}")
    return path

dfs_result = dfs_attack_path(G, 'Internet', 'DBServer', 'Web Attack - SQL Injection')

In [ ]:
def hill_climbing_response(attack_type, compromised_nodes):
   
    response_actions = {
        'Block_IP':          {'DDoS': 0.7, 'Portscan': 0.9, 'Web Attack - SQL Injection': 0.5, 'Botnet': 0.6},
        'Rate_Limiting':     {'DDoS': 0.9, 'Portscan': 0.6, 'Web Attack - SQL Injection': 0.4, 'Botnet': 0.5},
        'Isolate_Node':      {'DDoS': 0.6, 'Portscan': 0.5, 'Web Attack - SQL Injection': 0.8, 'Botnet': 0.9},
        'Patch_Firewall':    {'DDoS': 0.5, 'Portscan': 0.7, 'Web Attack - SQL Injection': 0.9, 'Botnet': 0.7},
        'Deploy_Honeypot':   {'DDoS': 0.3, 'Portscan': 0.8, 'Web Attack - SQL Injection': 0.6, 'Botnet': 0.8},
        'Traffic_Scrubbing': {'DDoS': 0.95,'Portscan': 0.4, 'Web Attack - SQL Injection': 0.3, 'Botnet': 0.4},
    }

    print(f"{'='*55}")
    print(f"HILL CLIMBING RESPONSE OPTIMIZER")
    print(f"Attack Detected: {attack_type}")
    print(f"Compromised Nodes: {compromised_nodes}")
    print(f"{'='*55}")

    import random
    current_action = random.choice(list(response_actions.keys()))
    
    current_score = response_actions[current_action].get(attack_type, 0.3)
    
    print(f"\nInitial action: {current_action} (score: {current_score})")
    print(f"\nClimbing...")

    iteration = 0
    while True:
        iteration += 1
        improved = False
        
        for action, scores in response_actions.items():
            score = scores.get(attack_type, 0.3)
            if score > current_score:
                print(f"  Step {iteration}: {current_action} ({current_score}) → {action} ({score})")
                current_action = action
                current_score = score
                improved = True
                break
        
        if not improved:
            break

    print(f"\n{'='*55}")
    print(f"OPTIMAL RESPONSE: {current_action}")
    print(f"Effectiveness Score: {current_score}")
    print(f"{'='*55}")
    return current_action, current_score

hill_climbing_response('DDoS', bfs_result)

In [ ]:
def full_ids_pipeline(sample_index):
    print(f"{'='*60}")
    print(f"FULL IDS PIPELINE - SAMPLE {sample_index}")
    print(f"{'='*60}")

    sample = X_final_combined[sample_index].reshape(1, -1)
    prediction = final_model_weighted.predict(sample)[0]
    attack_label = le_combined.classes_[prediction]
    confidence = final_model_weighted.predict_proba(sample)[0][prediction]

    print(f"\nSTAGE 1 -- ML CLASSIFICATION")
    print(f"  Predicted Attack: {attack_label}")
    print(f"  Confidence: {confidence:.4f}")
    print(f"  Actual Label: {le_combined.classes_[y_combined_encoded[sample_index]]}")

    if attack_label != 'BENIGN':
        print(f"\nSTAGE 2 -- BFS BLAST RADIUS")
        bfs_nodes = bfs_blast_radius(G, 'Internet', 'Router', attack_label)

        print(f"\nSTAGE 3 -- DFS ATTACK PATH")
        if 'SQL' in attack_label or 'Web' in attack_label:
            target = 'DBServer'
        elif 'Botnet' in attack_label:
            target = 'FileServer'
        else:
            target = 'WebServer'

        dfs_path = dfs_attack_path(G, 'Internet', target, attack_label)

        print(f"\nSTAGE 4 -- HILL CLIMBING RESPONSE")
        optimal_action, score = hill_climbing_response(attack_label, bfs_nodes)

        print(f"\n{'='*60}")
        print(f"PIPELINE SUMMARY")
        print(f"  Attack Type: {attack_label}")
        print(f"  Confidence: {confidence:.4f}")
        print(f"  Nodes at Risk: {len(bfs_nodes)}")
        print(f"  Attack Path Depth: {len(dfs_path)-1} hops")
        print(f"  Optimal Response: {optimal_action} (score: {score})")
        print(f"{'='*60}")
    else:
        print(f"\n  Traffic is BENIGN -- no action required")

import random
print("Testing pipeline on attack samples (using final full-data model)...\n")

ddos_idx = list(y_combined_encoded).index(le_combined.transform(['DDoS'])[0])
full_ids_pipeline(ddos_idx)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, ax = plt.subplots(figsize=(14, 20))
ax.set_xlim(0, 14)
ax.set_ylim(0, 20)
ax.axis('off')
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

COLORS = {
    'input':      '#1e3a5f',
    'preprocess': '#1a4a3a',
    'iso':        '#3d1f6e',
    'xgb':        '#4a1f6e',
    'bfs':        '#1a3d5c',
    'dfs':        '#1a3d5c',
    'hc':         '#1a3d5c',
    'output':     '#6e1f1f',
    'benign':     '#1a3a1a',
}

def draw_box(ax, cx, cy, w, h, title, subtitle=None,
             facecolor='#1e3a5f', bordercolor='#2196f3',
             title_color='#ffffff', sub_color='#aaaaaa',
             fontsize=10, subfontsize=8):
    box = FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                         boxstyle="round,pad=0.25",
                         facecolor=facecolor, edgecolor=bordercolor,
                         linewidth=2, zorder=3)
    ax.add_patch(box)
    if subtitle:
        ax.text(cx, cy+0.18, title, ha='center', va='center',
                fontsize=fontsize, fontweight='bold',
                color=title_color, zorder=4)
        ax.text(cx, cy-0.22, subtitle, ha='center', va='center',
                fontsize=subfontsize, color=sub_color,
                zorder=4, style='italic')
    else:
        ax.text(cx, cy, title, ha='center', va='center',
                fontsize=fontsize, fontweight='bold',
                color=title_color, zorder=4)

def draw_arrow(ax, x1, y1, x2, y2, color='#555577'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->',
                               color=color, lw=2.0,
                               mutation_scale=18), zorder=2)

def draw_diamond(ax, cx, cy, w, h, text,
                 facecolor='#2a2a4a', bordercolor='#7777aa'):
    xs = [cx, cx+w/2, cx, cx-w/2]
    ys = [cy+h/2, cy, cy-h/2, cy]
    diamond = plt.Polygon(list(zip(xs, ys)),
                          facecolor=facecolor,
                          edgecolor=bordercolor,
                          linewidth=2, zorder=3)
    ax.add_patch(diamond)
    ax.text(cx, cy, text, ha='center', va='center',
            fontsize=9, fontweight='bold',
            color='#ffffff', zorder=4)

def draw_section_label(ax, x, y, text, color):
    ax.text(x, y, text, ha='left', va='center',
            fontsize=7.5, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3',
                     facecolor=color+'22',
                     edgecolor=color, linewidth=0.8), zorder=5)

ax.text(7, 19.4, 'Hybrid IDS System Architecture',
        ha='center', fontsize=16, fontweight='bold',
        color='#ffffff', zorder=5)
ax.text(7, 19.0,
        'Isolation Forest  .  XGBoost  .  BFS  .  DFS  .  Hill Climbing',
        ha='center', fontsize=9, color='#7799bb',
        zorder=5, style='italic')

draw_section_label(ax, 0.15, 17.8, '(1) DATA',     '#4caf50')
draw_section_label(ax, 0.15, 15.5, '(2) ML STAGE 1','#9c27b0')
draw_section_label(ax, 0.15, 13.2, '(3) ML STAGE 2','#ce93d8')
draw_section_label(ax, 0.15, 10.5, '(4) GRAPH',    '#2196f3')
draw_section_label(ax, 0.15, 7.2,  '(5) RESPONSE', '#f44336')

draw_box(ax, 7, 18.3, 7, 0.9,
         'Network Traffic Input',
         'CICIDS-2017  .  2,099,971 flows  .  27 attack classes  .  86 features',
         facecolor=COLORS['input'], bordercolor='#2196f3',
         fontsize=11, subfontsize=8)
draw_arrow(ax, 7, 17.85, 7, 17.2, color='#2196f3')

draw_box(ax, 7, 16.9, 7.5, 0.85,
         'Preprocessing  &  Feature Engineering',
         'Strip spaces  .  Drop IP/Timestamp  .  Label Encoding  .  Z-score normalization',
         facecolor=COLORS['preprocess'], bordercolor='#4caf50',
         fontsize=10, subfontsize=8)
draw_arrow(ax, 7, 16.47, 7, 15.82, color='#4caf50')

draw_box(ax, 7, 15.5, 7.5, 0.85,
         'Stage 1  -  Isolation Forest  (Unsupervised)',
         '100 estimators  .  contamination=0.25  .  Anomaly score as 86th feature',
         facecolor=COLORS['iso'], bordercolor='#9c27b0',
         fontsize=10, subfontsize=8)

ax.annotate('Anomaly\nscore added\nas feature',
            xy=(10.8, 15.5), xytext=(12.0, 15.5),
            fontsize=7.5, color='#9c27b0',
            ha='center', va='center',
            arrowprops=dict(arrowstyle='<-',
                           color='#9c27b0', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3',
                     facecolor='#3d1f6e',
                     edgecolor='#9c27b0', linewidth=0.8),
            zorder=5)

draw_arrow(ax, 7, 15.07, 7, 14.42, color='#9c27b0')

draw_box(ax, 7, 14.1, 8.5, 0.85,
         'Stage 2  -  XGBoost  (Supervised Ensemble)',
         'Stratified K-Fold (k=5)  .  L1/L2 regularization  .  Balanced class weights  .  27 classes',
         facecolor=COLORS['xgb'], bordercolor='#ce93d8',
         fontsize=10, subfontsize=8)

ax.annotate('99.81%\naccuracy',
            xy=(10.8, 14.1), xytext=(12.2, 14.1),
            fontsize=7.5, color='#ce93d8',
            ha='center', va='center',
            arrowprops=dict(arrowstyle='<-',
                           color='#ce93d8', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3',
                     facecolor='#4a1f6e',
                     edgecolor='#ce93d8', linewidth=0.8),
            zorder=5)

draw_arrow(ax, 7, 13.67, 7, 13.05, color='#ce93d8')

draw_diamond(ax, 7, 12.65, 4, 0.8, 'Attack detected?',
             facecolor='#1a1a2e', bordercolor='#7777cc')

ax.plot([5.0, 2.5], [12.65, 12.65],
        color='#4caf50', lw=1.8, zorder=2)
ax.annotate('', xy=(2.5, 12.25), xytext=(2.5, 12.65),
            arrowprops=dict(arrowstyle='->',
                           color='#4caf50', lw=1.8), zorder=2)
ax.text(3.7, 12.8, 'No', ha='center', fontsize=8.5,
        color='#4caf50', fontweight='bold')

draw_box(ax, 2.0, 11.9, 2.8, 0.6,
         'BENIGN', 'No action required',
         facecolor=COLORS['benign'], bordercolor='#4caf50',
         fontsize=9, subfontsize=7.5)

draw_arrow(ax, 7, 12.25, 7, 11.62, color='#f44336')
ax.text(7.35, 11.92, 'Yes', ha='left', fontsize=8.5,
        color='#f44336', fontweight='bold')

draw_box(ax, 7, 11.3, 8.5, 0.55,
         'Graph-Based Network Analysis  -  G = (V, E)',
         '9 nodes  .  11 directed edges  .  Enterprise topology model',
         facecolor='#0d1f3a', bordercolor='#1565c0',
         fontsize=9.5, subfontsize=7.5)
draw_arrow(ax, 7, 11.02, 7, 10.52, color='#2196f3')

ax.plot([7, 2.3],  [10.52, 10.52], color='#2196f3', lw=1.8, zorder=2)
ax.plot([7, 7],    [10.52, 10.52], color='#2196f3', lw=1.8, zorder=2)
ax.plot([7, 11.7], [10.52, 10.52], color='#2196f3', lw=1.8, zorder=2)
ax.annotate('', xy=(2.3,  10.1), xytext=(2.3,  10.52),
            arrowprops=dict(arrowstyle='->', color='#2196f3', lw=1.8), zorder=2)
ax.annotate('', xy=(7,    10.1), xytext=(7,    10.52),
            arrowprops=dict(arrowstyle='->', color='#2196f3', lw=1.8), zorder=2)
ax.annotate('', xy=(11.7, 10.1), xytext=(11.7, 10.52),
            arrowprops=dict(arrowstyle='->', color='#2196f3', lw=1.8), zorder=2)

draw_box(ax, 2.3, 9.6, 3.8, 0.9,
         'BFS  -  Blast Radius',
         'Level-by-level traversal\nRisk score R(v) = 1.0 - 0.2 x d(v)',
         facecolor=COLORS['bfs'], bordercolor='#1976d2',
         fontsize=9.5, subfontsize=7.5)

draw_box(ax, 7, 9.6, 3.8, 0.9,
         'DFS  -  Kill Chain',
         'Recursive depth traversal\nFull compromise path traced',
         facecolor=COLORS['dfs'], bordercolor='#1976d2',
         fontsize=9.5, subfontsize=7.5)

draw_box(ax, 11.7, 9.6, 3.8, 0.9,
         'Hill Climbing  -  Response',
         'Iterative score optimization\nConverges to optimal action',
         facecolor=COLORS['hc'], bordercolor='#1976d2',
         fontsize=9.5, subfontsize=7.5)

ax.plot([2.3,  2.3],  [9.15, 8.5], color='#2196f3', lw=1.8, zorder=2)
ax.plot([7,    7],    [9.15, 8.5], color='#2196f3', lw=1.8, zorder=2)
ax.plot([11.7, 11.7], [9.15, 8.5], color='#2196f3', lw=1.8, zorder=2)
ax.plot([2.3,  11.7], [8.5,  8.5], color='#2196f3', lw=1.8, zorder=2)
draw_arrow(ax, 7, 8.5, 7, 7.92, color='#f44336')

draw_box(ax, 7, 7.55, 9, 0.9,
         'Situational Awareness Report',
         'Attack type  .  Confidence score  .  Nodes at risk  .  Kill chain  .  Optimal response',
         facecolor=COLORS['output'], bordercolor='#f44336',
         fontsize=11, subfontsize=8.5, title_color='#ff8a80')

metrics = [
    (2.5,  6.5, '99.81%',     'Overall accuracy'),
    (5.0,  6.5, '27 classes', 'Attack categories'),
    (7.5,  6.5, '2M+ flows',  'Training samples'),
    (10.0, 6.5, 'Top 5',      'Anomaly score rank'),
    (12.5, 6.5, 'K=5 folds',  'Cross-validation'),
]
for mx, my, val, label in metrics:
    draw_box(ax, mx, my, 2.2, 0.75, val, label,
             facecolor='#1a1a2e', bordercolor='#333355',
             fontsize=10, subfontsize=7.5,
             title_color='#64b5f6', sub_color='#888888')

novel = FancyBboxPatch((0.5, 5.5), 13, 0.7,
                       boxstyle="round,pad=0.2",
                       facecolor='#1a0a0a', edgecolor='#f44336',
                       linewidth=1.5, linestyle='--', zorder=3)
ax.add_patch(novel)
ax.text(7, 5.95, 'Key Design Feature',
        ha='center', va='center', fontsize=9,
        fontweight='bold', color='#f44336', zorder=4)
ax.text(7, 5.68,
        'Unified offline pipeline: ML anomaly scoring + ensemble classification + post-detection graph reasoning',
        ha='center', va='center', fontsize=8,
        color='#aaaaaa', zorder=4, style='italic')

plt.tight_layout(pad=0.5)
plt.savefig('pipeline_flowchart.png', dpi=300,
            bbox_inches='tight', facecolor='#0d1117',
            edgecolor='none')
plt.show()
print("Flowchart saved as pipeline_flowchart.png!")

In [ ]:
from sklearn.model_selection import train_test_split
import xgboost as xgb

print("Running ablation study...")
print("XGBoost on feature set WITHOUT the Isolation Forest anomaly score")
print("Using a STRATIFIED 500,000-sample subset, matching what the paper claims.")
print("Using XGBoost's native Booster API to avoid the same rare-class-missing-from-fold issue.")
print("="*55)

NUM_CLASSES = len(le_combined.classes_)

SUBSET_SIZE = 500_000
if len(X_combined_raw) > SUBSET_SIZE:
    X_ablation_pool, _, y_ablation_pool, _ = train_test_split(
        X_combined_raw, y_combined_encoded,
        train_size=SUBSET_SIZE, stratify=y_combined_encoded, random_state=42
    )
else:
    X_ablation_pool, y_ablation_pool = X_combined_raw, y_combined_encoded

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results_ablation = []

xgb_params_ablation = {
    'objective': 'multi:softprob',
    'num_class': NUM_CLASSES,
    'max_depth': 6,
    'eta': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'seed': 42,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda'
}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_ablation_pool, y_ablation_pool)):
    X_train_raw, X_test_raw = X_ablation_pool[train_idx], X_ablation_pool[test_idx]
    y_train, y_test = y_ablation_pool[train_idx], y_ablation_pool[test_idx]

    scaler_fold = StandardScaler()
    X_train = scaler_fold.fit_transform(X_train_raw)
    X_test = scaler_fold.transform(X_test_raw)

    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weights)
    dtest = xgb.DMatrix(X_test, label=y_test)

    booster_ablation = xgb.train(xgb_params_ablation, dtrain, num_boost_round=100)
    y_pred_proba = booster_ablation.predict(dtest)
    y_pred = y_pred_proba.argmax(axis=1)

    report = classification_report(y_test, y_pred,
                                    labels=list(range(NUM_CLASSES)),
                                    target_names=le_combined.classes_,
                                    output_dict=True, zero_division=0)
    fold_results_ablation.append(report)
    print(f"Fold {fold+1} complete")

print("\nAblation study complete!")
print("IMPORTANT: rerun this and update the paper's ablation accuracy/std numbers --")
print("the previous numbers in the paper came from the FULL dataset (a code/text mismatch),")
print("not this 500k stratified subset.")


In [ ]:
print("="*50)
print("ABLATION RESULTS — XGBoost WITHOUT IF score")
print("="*50)

accuracies = [f['accuracy'] for f in fold_results_ablation]
f1s = [f['weighted avg']['f1-score'] for f in fold_results_ablation]

print(f"Mean Accuracy: {np.mean(accuracies):.4f}")
print(f"Std Accuracy:  {np.std(accuracies):.4f}")
print(f"Mean F1:       {np.mean(f1s):.4f}")
print(f"Std F1:        {np.std(f1s):.4f}")

print(f"\n{'='*50}")
print(f"COMPARISON")
print(f"{'='*50}")
print(f"Full model (WITH IF score):    0.9981")
print(f"Ablation  (WITHOUT IF score):  {np.mean(accuracies):.4f}")
print(f"Improvement from IF score:    +{0.9981 - np.mean(accuracies):.4f}")

In [ ]:
print("Extracting missing Table I values...")
print("="*55)

classes_needed = ['DoS GoldenEye', 
                  'Infiltration - Portscan',
                  'Web Attack - Brute Force']

for cls in classes_needed:
    try:
        avg_prec = np.mean([f[cls]['precision'] for f in fold_results_weighted])
        avg_rec  = np.mean([f[cls]['recall']    for f in fold_results_weighted])
        avg_f1   = np.mean([f[cls]['f1-scor e']  for f in fold_results_weighted])
        print(f"\n{cls}:")
        print(f"  Precision: {avg_prec:.4f}")
        print(f"  Recall:    {avg_rec:.4f}")
        print(f"  F1:        {avg_f1:.4f}")
    except KeyError:
        matches = [k for k in fold_results_weighted[0].keys() 
                   if any(x in k for x in ['Golden', 'Infiltr', 'Brute'])]
        print(f"\n{cls}: exact key is → {matches}")

In [ ]:
import xgboost as xgb
import numpy as np

print("Running ablation: full pipeline WITHOUT Src Port / Dst Port")
print("="*55)

port_cols = [c for c in X_combined.columns if c in ('Src Port', 'Dst Port')]
print("Dropping columns:", port_cols)
port_indices = [X_combined.columns.get_loc(c) for c in port_cols]
X_noports_raw = np.delete(X_combined_raw, port_indices, axis=1)
print("Shape before:", X_combined_raw.shape, "-> after dropping ports:", X_noports_raw.shape)

NUM_CLASSES = len(le_combined.classes_)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results_noports = []

xgb_params_noports = {
    'objective': 'multi:softprob',
    'num_class': NUM_CLASSES,
    'max_depth': 6,
    'eta': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'seed': 42,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda'
}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_noports_raw, y_combined_encoded)):
    X_train_raw, X_test_raw = X_noports_raw[train_idx], X_noports_raw[test_idx]
    y_train, y_test = y_combined_encoded[train_idx], y_combined_encoded[test_idx]

    scaler_fold = StandardScaler()
    X_train_scaled = scaler_fold.fit_transform(X_train_raw)
    X_test_scaled = scaler_fold.transform(X_test_raw)

    iso_fold = IsolationForest(n_estimators=100, contamination=0.25, random_state=42, n_jobs=-1)
    iso_fold.fit(X_train_scaled)
    train_iso_scores = iso_fold.score_samples(X_train_scaled).reshape(-1, 1)
    test_iso_scores = iso_fold.score_samples(X_test_scaled).reshape(-1, 1)

    X_train_final = np.hstack([X_train_scaled, train_iso_scores])
    X_test_final = np.hstack([X_test_scaled, test_iso_scores])

    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    dtrain = xgb.DMatrix(X_train_final, label=y_train, weight=sample_weights)
    dtest = xgb.DMatrix(X_test_final, label=y_test)

    booster = xgb.train(xgb_params_noports, dtrain, num_boost_round=100)
    y_pred = booster.predict(dtest).argmax(axis=1)

    report = classification_report(y_test, y_pred,
                                    labels=list(range(NUM_CLASSES)),
                                    target_names=le_combined.classes_,
                                    output_dict=True, zero_division=0)
    fold_results_noports.append(report)
    print(f"Fold {fold+1} complete")

acc_noports = np.mean([f['accuracy'] for f in fold_results_noports])
print(f"\n{'='*55}")
print(f"Full pipeline WITH ports:    0.9981")
print(f"Full pipeline WITHOUT ports: {acc_noports:.4f}")
print(f"Difference: {acc_noports - 0.9981:+.4f}")

In [ ]:
import numpy as np

print("Checking whether the anomaly score correlates with weak classes or misclassifications")
print("="*55)

anomaly_scores = X_final_combined[:, -1]
y_pred_full = final_model_weighted.predict(X_final_combined)
y_true_full = y_combined_encoded

correct = (y_pred_full == y_true_full)

print(f"Mean anomaly score - correctly classified: {anomaly_scores[correct].mean():.4f}")
print(f"Mean anomaly score - misclassified:        {anomaly_scores[~correct].mean():.4f}")

weak_classes = ['Web Attack - SQL Injection', 'Web Attack - SQL Injection - Attempted',
                 'Web Attack - XSS - Attempted', 'Web Attack - Brute Force - Attempted',
                 'Heartbleed', 'Infiltration', 'FTP-Patator - Attempted']

print("\nMean anomaly score by class (sorted lowest to highest):")
class_scores = {}
for i, cls in enumerate(le_combined.classes_):
    mask = (y_true_full == i)
    if mask.sum() > 0:
        class_scores[cls] = anomaly_scores[mask].mean()

for cls, score in sorted(class_scores.items(), key=lambda x: x[1]):
    tag = "  <-- WEAK CLASS" if cls in weak_classes else ""
    print(f"  {cls:45s}: {score:.4f}{tag}")

overall_mean = anomaly_scores.mean()
weak_mean = np.mean([class_scores[c] for c in weak_classes if c in class_scores])
print(f"\nOverall mean: {overall_mean:.4f}   Weak-class mean: {weak_mean:.4f}")

In [ ]:
import contextlib
import io
import networkx as nx
import random
import time
import statistics
import math
import matplotlib.pyplot as plt

def generate_synthetic_topology(n_nodes, seed):
  
    rng = random.Random(seed)
    G = nx.DiGraph()
    G.add_node('Internet')

    n_firewall = max(1, n_nodes // 50)
    n_core = max(1, n_nodes // 25)
    n_dist = max(1, n_nodes // 8)
    n_leaf = max(1, n_nodes - 1 - n_firewall - n_core - n_dist)

    def add_layer(prefix, count):
        nodes = [f"{prefix}{i}" for i in range(count)]
        G.add_nodes_from(nodes)
        return nodes

    fw = add_layer('Firewall', n_firewall)
    core = add_layer('Core', n_core)
    dist = add_layer('Dist', n_dist)
    leaf = add_layer('Leaf', n_leaf)

    layers = [['Internet'], fw, core, dist, leaf]
    for i in range(1, len(layers)):
        upper = layers[i-1]
        for node in layers[i]:
            k = rng.randint(1, min(3, len(upper)))
            for p in rng.sample(upper, k):
                G.add_edge(p, node)
    return G, leaf


sizes = [9, 25, 50, 100, 250, 500, 1000, 2000]
n_topologies = 10
n_timing_reps = 5

print(f"{'Nodes':>8} {'Edges(avg)':>11} {'BFS mean(ms)':>13} {'DFS mean(ms)':>13} {'HC mean(ms)':>12}")
summary = []
for n in sizes:
    bfs_all, dfs_all, hc_all, edge_counts = [], [], [], []
    for topo_i in range(n_topologies):
        G, leaf_nodes = generate_synthetic_topology(n, seed=1000*n + topo_i)
        rng = random.Random(2000*n + topo_i)
        target = rng.choice(leaf_nodes)
        edge_counts.append(G.number_of_edges())

        for _ in range(n_timing_reps):
            with contextlib.redirect_stdout(io.StringIO()):
                t0 = time.perf_counter(); bfs_blast_radius(G, 'Internet', target, 'DDoS'); bfs_all.append(time.perf_counter()-t0)
            with contextlib.redirect_stdout(io.StringIO()):
                t0 = time.perf_counter(); dfs_attack_path(G, 'Internet', target, 'Web Attack - SQL Injection'); dfs_all.append(time.perf_counter()-t0)
            with contextlib.redirect_stdout(io.StringIO()):
                t0 = time.perf_counter(); hill_climbing_response('DDoS', []); hc_all.append(time.perf_counter()-t0)

    bfs_mean, dfs_mean, hc_mean = statistics.mean(bfs_all)*1000, statistics.mean(dfs_all)*1000, statistics.mean(hc_all)*1000
    avg_edges = statistics.mean(edge_counts)
    print(f"{n:>8} {avg_edges:>11.1f} {bfs_mean:>13.4f} {dfs_mean:>13.4f} {hc_mean:>12.4f}")
    summary.append((n, avg_edges, bfs_mean, dfs_mean, hc_mean))

ns = [s[0] for s in summary]; bfs_times = [s[2] for s in summary]
log_n = [math.log(n) for n in ns]; log_t = [math.log(t) for t in bfs_times]
mean_x, mean_y = sum(log_n)/len(log_n), sum(log_t)/len(log_t)
slope = sum((x-mean_x)*(y-mean_y) for x,y in zip(log_n,log_t)) / sum((x-mean_x)**2 for x in log_n)
print(f"\nEstimated BFS scaling exponent k = {slope:.2f} (k~1 = linear O(V), k~2 = quadratic O(V^2))")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(ns, [s[2] for s in summary], marker='o', label='BFS Blast Radius', linewidth=2)
ax.plot(ns, [s[3] for s in summary], marker='s', label='DFS Kill Chain', linewidth=2)
ax.plot(ns, [s[4] for s in summary], marker='^', label='Hill Climbing Response', linewidth=2)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Number of Nodes in Topology'); ax.set_ylabel('Mean Runtime (ms)')
ax.set_title('Algorithm Runtime Scaling on Synthetic Enterprise Topologies')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('graph_scaling.png', dpi=300)
plt.show()

In [ ]:
for row in summary:
    print(row)

In [ ]:
import numpy as np

fold_accuracies = [f['accuracy'] for f in fold_results_weighted]
mean_acc = np.mean(fold_accuracies)
std_acc = np.std(fold_accuracies, ddof=1) 

print("Per-fold accuracies:", [f"{a:.4f}" for a in fold_accuracies])
print(f"Mean: {mean_acc:.4f}")
print(f"Std Dev: {std_acc:.4f}")
print(f"Report as: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")